# Named Entity Recognition

Pull the names out. Sounds easy util you deal with ambiguous boundaries, nested entities, and domain jargon.

## Problem Definition

Pass

## Basic Concept

### BIO tagging (BILOU)

turns entity extraction into a sequence-labeling problem. Label each token with `B-TYPE` (beginning of entity), `I-TYPE` (inside entity), or `O` (outside any entity).

```
Apple    B-ORG
sued     O
Google   B-ORG
over     O
its      O
iPhone   B-PRODUCT
search   O
deal     O
in       O
the      O
US       B-GPE
.        O
```

Multi-token entities chain: `New B-GPE`, `York I-GPE`, `City I-GPE`. A model that understands BIO can extract arbitrary spans.


### Architecture Progressing



In [2]:
import pandas as pd

df = pd.DataFrame({
    "Progress": ["Rule-based", "Hidden Markov Model", "Conditional Random Field", "BiLSTM-CRF", "Transformer-based"],
    "Basics": ["Regex", "Emission & Transition Probabilities", "Discriminative", "Neural features", "Fine-tune BERT with a token specification head"],
    "Problems": ["Zero coverage on new ones", "", "", "", "Most Compute"]
})

df

,Progress,Basics,Problems
0,Rule-based,Regex,Zero coverage on new ones
1,Hidden Markov Model,Emission & Transition Probabilities,
2,Conditional Random Field,Discriminative,
3,BiLSTM-CRF,Neural features,
4,Transformer-based,Fine-tune BERT with a token specification head,Most Compute


# Build your Own

## BIO tagging helpers

In [4]:
def spans_to_bio(tokens, spans):
    labels = ["0"] * len(tokens)
    for start, end, label in spans:
        labels[start] = f"B-{label}"
        for i in range(start + 1, end):
            labels[i] = f"I-{label}"
    return labels

def bio_to_spans(tokens, labels):
    spans = []
    current = None
    for i, label in enumerate(labels):
        if label.startswith("B-"):
            if current:
                spans.append(current)
            current = (i, i + 1, label[2:])
        elif label.startswith("I-") and current and current[2] == label[2:]:
            current = (current[0], i + 1, current[2])
        else:
            if current:
                spans.append(current)
                current = None

    if current:
        spans.append(current)
    return spans

tokens = ["Apple", "sued", "Google", "over", "iPhone", "sales", "."]
labels = ["B-ORG", "O", "B-ORG", "O", "B-PRODUCT", "O", "O"]
spans = bio_to_spans(tokens, labels)
print(spans)

labels = spans_to_bio(tokens, spans)
print(labels)




[(0, 1, 'ORG'), (2, 3, 'ORG'), (4, 5, 'PRODUCT')]
['B-ORG', '0', 'B-ORG', '0', 'B-PRODUCT', '0', '0']


## Hand-crafted features

In [6]:
def token_features(token, prev_token, next_token):
    return {
        "lower": token.lower(),
        "is_upper": token.isupper(),
        "is_title": token.istitle(),
        "has_digit": any(c.isdigit() for c in token),
        "suffix_3": token[-3:].lower(),
        "shape": word_shape(token),
        "prev_lower": prev_token.lower() if prev_token else "<BOS>",
        "next_lower": next_token.lower() if next_token else "<EOS>",
    }

def word_shape(word):
    out = []
    for c in word:
        if c.isupper():
            out.append("X")
        elif c.islower():
            out.append("x")
        elif c.isdigit():
            out.append("d")
        else:
            out.append(c)
    return "".join(out)

print(word_shape("iPhone"))
print(word_shape("USA-2024"))

xXxxxx
XXX-dddd


## Simple Rule base + dictionary baseline

In [ ]:
ORG_GAZETTEER = {"Apple", "Google", "Microsoft", "OpenAI", "Meta", "Amazon", "Netflix"}
GPE_GAZETTEER = {"US", "USA", "UK", "India", "Germany", "France"}
PRODUCT_GAZETTEER = {"iPhone", "Android", "Windows", "ChatGPT", "Claude"}

def rule_based_ner(tokens):
    labels = []
    for token in tokens:
        if token in ORG_GAZETTEER:
            labels.append("B-ORG")
        elif token in GPE_GAZETTEER:
            labels.append("B-GPE")
        elif token in PRODUCT_GAZETTEER:
            labels.append("B-PRODUCT")
        else:
            labels.append("O")
    return labels